# Features V2 — Ingeniería de Variables Avanzada

## Caso Práctico - Empresa de Telecomunicaciones
## Prácticas Aplicadas 2026

---

## Motivación

En el notebook anterior (`modelo_features`) añadimos variables de tendencia y el AUC subió de **0.690 a 0.701**.

En este notebook añadimos tres variables adicionales:

1. **`critica_pendiente_lag1`** — incidencia de Facturación/Baja sin resolver el mes anterior.
2. **`sentimiento_zonal_lag1`** — sentimiento medio de encuestas en la zona el mes anterior.
3. **`ratio_estres_lag1`** — combina días de retraso en el pago con la calidad de red: `(dias_retraso+1)/(calidad_global+1)`.

Principio anti-leakage: todas las variables usan información del mes anterior (lag 1).


## 1. Librerías


In [ ]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, average_precision_score,
    RocCurveDisplay, PrecisionRecallDisplay
)

try:
    from xgboost import XGBClassifier
    XGBOOST_OK = True
except ImportError:
    XGBOOST_OK = False
    print('XGBoost no instalado')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid', context='notebook')

ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.load import load_all
from src.clean import clean_all

RANDOM_STATE = 42
print('Librerias cargadas')


Librerias cargadas


## 2. Carga de datos


In [ ]:
data  = load_all()
clean = clean_all(data, save=False)

clientes  = clean['clientes']
churn     = clean['churn']
factura   = clean['facturacion']
soporte   = clean['soporte']
calidad   = clean['calidad']
encuestas = clean['encuestas']

print('Datos cargados')


Datos cargados


---
## 3. Construccion del panel

Mantenemos todas las variables de `modelo_features` y añadimos las tres nuevas.


In [ ]:
# Facturacion mensual base
factura['mes'] = factura['fecha'].dt.to_period('M').dt.to_timestamp()

factura_mes = factura.groupby(['cliente_id', 'mes']).agg(
    importe_mes       = ('importe_total',        'sum'),
    impago_mes        = ('impago_flag',           'max'),
    dias_retraso_mes  = ('dias_retraso_pago',     'max'),
    stress_mes        = ('stress_calidad_lag',    'mean'),
    consumo_extra_mes = ('consumo_extra',         'sum'),
    variacion_consumo = ('variacion_consumo_pct', 'mean'),
    descuento_mes     = ('descuento_aplicado',    'sum'),
).reset_index().sort_values(['cliente_id', 'mes'])

print(f'Facturacion mensual: {len(factura_mes):,} registros')


Facturacion mensual: 326,816 registros


In [ ]:
# Features de tendencia (heredadas de modelo_features)

# 1. Tendencia de importe: diferencia entre t-1 y t-3
factura_mes['tendencia_importe'] = (
    factura_mes.groupby('cliente_id')['importe_mes']
    .transform(lambda x: x.shift(1) - x.shift(3))
)

# 2. Tendencia de stress de red: diferencia entre t-1 y t-3
factura_mes['tendencia_stress'] = (
    factura_mes.groupby('cliente_id')['stress_mes']
    .transform(lambda x: x.shift(1) - x.shift(3))
)

# 3. Racha de impagos consecutivos
impago_lag = factura_mes.groupby('cliente_id')['impago_mes'].shift(1).fillna(0)
grupos_racha = (
    impago_lag != impago_lag.groupby(factura_mes['cliente_id']).shift(1)
).cumsum()
factura_mes['racha_impagos'] = (
    impago_lag * factura_mes.groupby(grupos_racha).cumcount().add(1)
)

# 4. Flag de desengagement: 2+ meses sin consumo extra
factura_mes['sin_consumo_2m'] = (
    factura_mes.groupby('cliente_id')['consumo_extra_mes']
    .transform(lambda x: ((x.shift(1) <= 0) & (x.shift(2) <= 0)).astype(int))
)

# 5. Rolling 3 meses
for col in ['impago_mes', 'stress_mes', 'variacion_consumo', 'dias_retraso_mes']:
    factura_mes[col + '_roll3'] = (
        factura_mes.groupby('cliente_id')[col]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    )

n_nulos_imp = factura_mes['tendencia_importe'].isnull().sum()
n_nulos_str = factura_mes['tendencia_stress'].isnull().sum()
print('Features de tendencia creadas:')
print(f'  tendencia_importe  - nulos: {n_nulos_imp:,}')
print(f'  tendencia_stress   - nulos: {n_nulos_str:,}')
print(f'  racha_impagos      - nulos: 0')
print(f'  sin_consumo_2m     - nulos: 0')


Features de tendencia creadas:
  tendencia_importe  - nulos: 29,915
  tendencia_stress   - nulos: 29,915
  racha_impagos      - nulos: 0
  sin_consumo_2m     - nulos: 0


In [ ]:
# Satisfaccion historica de soporte (ultimos 6 meses con interaccion)
soporte['mes_dt'] = soporte['mes'].dt.to_period('M').dt.to_timestamp()

sop_hist = (
    soporte.groupby(['cliente_id', 'mes_dt'])['satisfaccion_post']
    .mean()
    .reset_index()
    .sort_values(['cliente_id', 'mes_dt'])
)
sop_hist['satisfaccion_hist6'] = (
    sop_hist.groupby('cliente_id')['satisfaccion_post']
    .transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean())
)
sop_hist = sop_hist.rename(columns={'mes_dt': 'mes'})

soporte_mes = (
    soporte.groupby(['cliente_id', 'mes_dt']).agg(
        n_interacciones_mes = ('interaccion_id', 'count'),
        n_baja_mes = ('motivo', lambda x: (x == 'Baja / portabilidad').sum()),
    ).reset_index().rename(columns={'mes_dt': 'mes'})
)

n_cl  = sop_hist['cliente_id'].nunique()
n_nul = sop_hist['satisfaccion_hist6'].isnull().sum()
print(f'Satisfaccion hist6 - clientes con historial: {n_cl:,}')
print(f'Nulos en satisfaccion_hist6: {n_nul:,}')


Satisfaccion hist6 - clientes con historial: 9,973
Nulos en satisfaccion_hist6: 10,045


In [ ]:
# FEATURES V2

# 1. SOPORTE CRITICO PENDIENTE
motivos_criticos = ['Facturacion', 'Baja / portabilidad']
soporte['critica_pendiente'] = (
    soporte['motivo'].isin(motivos_criticos) & (soporte['resuelto'] == 0)
).astype(int)

soporte['mes_sop'] = soporte['mes'].dt.to_period('M').dt.to_timestamp()
sop_critico = (
    soporte.groupby(['cliente_id', 'mes_sop'])['critica_pendiente']
    .max().reset_index()
    .rename(columns={'mes_sop': 'mes'})
)
sop_critico_lag = sop_critico.copy()
sop_critico_lag['mes'] = sop_critico_lag['mes'] + pd.DateOffset(months=1)
sop_critico_lag = sop_critico_lag.rename(
    columns={'critica_pendiente': 'critica_pendiente_lag1'}
)

n_reg = len(sop_critico)
n_cri = sop_critico['critica_pendiente'].sum()
pct   = sop_critico['critica_pendiente'].mean() * 100
print(f'Soporte critico - registros cliente-mes: {n_reg:,}')
print(f'Con critica pendiente: {n_cri:,} ({pct:.1f}%)')

# 2. SENTIMIENTO ZONAL LAG1
clima_zonal = (
    encuestas.groupby(['zona_id', 'fecha'])['sent_text_latente']
    .mean().reset_index()
    .rename(columns={'sent_text_latente': 'sentimiento_zonal'})
)
clima_zonal['fecha'] = clima_zonal['fecha'].dt.to_period('M').dt.to_timestamp()
clima_zonal = clima_zonal.sort_values(['zona_id', 'fecha'])
clima_zonal['sentimiento_zonal_lag1'] = (
    clima_zonal.groupby('zona_id')['sentimiento_zonal'].shift(1)
)
clima_zonal = clima_zonal.rename(columns={'fecha': 'mes'})

n_zon = len(clima_zonal)
n_nul = clima_zonal['sentimiento_zonal_lag1'].isnull().sum()
print(f'Sentimiento zonal - zonas x meses: {n_zon:,}')
print(f'Nulos en lag1 (primer mes de cada zona): {n_nul:,}')

# 3. RATIO ESTRES ECONOMICO-RED
calidad_ratio = calidad[['zona_id', 'fecha', 'indice_calidad_global']].copy()
calidad_ratio['mes'] = calidad_ratio['fecha'].dt.to_period('M').dt.to_timestamp()

fac_ratio = factura_mes[['cliente_id', 'mes', 'dias_retraso_mes']].copy()
zona_cl   = clientes[['cliente_id', 'zona_id']]
fac_ratio = fac_ratio.merge(zona_cl, on='cliente_id', how='left')
fac_ratio = fac_ratio.merge(
    calidad_ratio[['zona_id', 'mes', 'indice_calidad_global']],
    on=['zona_id', 'mes'], how='left'
)
fac_ratio['ratio_estres'] = (
    (fac_ratio['dias_retraso_mes'].fillna(0) + 1) /
    (fac_ratio['indice_calidad_global'].fillna(50) + 1)
)

ratio_lag = fac_ratio[['cliente_id', 'mes', 'ratio_estres']].copy()
ratio_lag['mes'] = ratio_lag['mes'] + pd.DateOffset(months=1)
ratio_lag = ratio_lag.rename(columns={'ratio_estres': 'ratio_estres_lag1'})

print(f'Ratio estres - registros: {len(ratio_lag):,}')
print(f'Media ratio: {fac_ratio["ratio_estres"].mean():.3f}')
print(f'Max ratio: {fac_ratio["ratio_estres"].max():.1f}')


Soporte critico - registros cliente-mes: 179,108
Con critica pendiente: 26,657 (14.9%)
Sentimiento zonal - zonas x meses: 569
Nulos en lag1 (primer mes de cada zona): 30
Ratio estres - registros: 321,987
Media ratio: 0.115
Max ratio: 49.8


In [ ]:
# Construccion del panel completo
panel = churn.copy()
panel['mes'] = panel['fecha'].dt.to_period('M').dt.to_timestamp()

# Lag 1 - facturacion
cols_factura = [
    'importe_mes', 'impago_mes', 'dias_retraso_mes', 'stress_mes',
    'consumo_extra_mes', 'variacion_consumo',
    'tendencia_importe', 'tendencia_stress', 'racha_impagos', 'sin_consumo_2m',
    'impago_mes_roll3', 'stress_mes_roll3', 'variacion_consumo_roll3', 'dias_retraso_mes_roll3',
]
factura_lag = factura_mes[['cliente_id', 'mes'] + cols_factura].copy()
factura_lag['mes'] = factura_lag['mes'] + pd.DateOffset(months=1)
factura_lag = factura_lag.rename(columns={c: c + '_lag1' for c in cols_factura})
panel = panel.merge(factura_lag, on=['cliente_id', 'mes'], how='left')

# Lag 1 - soporte
soporte_lag = soporte_mes.copy()
soporte_lag['mes'] = soporte_lag['mes'] + pd.DateOffset(months=1)
soporte_lag = soporte_lag.rename(columns={
    'n_interacciones_mes': 'n_interacciones_mes_lag1',
    'n_baja_mes': 'n_baja_mes_lag1',
})
panel = panel.merge(soporte_lag, on=['cliente_id', 'mes'], how='left')
panel['n_interacciones_mes_lag1'] = panel['n_interacciones_mes_lag1'].fillna(0)
panel['n_baja_mes_lag1']          = panel['n_baja_mes_lag1'].fillna(0)

# Satisfaccion historica lag 1
sat_lag = sop_hist[['cliente_id', 'mes', 'satisfaccion_hist6']].copy()
sat_lag['mes'] = sat_lag['mes'] + pd.DateOffset(months=1)
panel = panel.merge(sat_lag, on=['cliente_id', 'mes'], how='left')

# Calidad de red lag 1
calidad['mes'] = calidad['fecha'].dt.to_period('M').dt.to_timestamp()
calidad_lag = calidad[[
    'zona_id', 'mes', 'indice_calidad_global', 'tasa_cortes_pct', 'cobertura_5g_pct'
]].copy()
calidad_lag['mes'] = calidad_lag['mes'] + pd.DateOffset(months=1)
calidad_lag = calidad_lag.rename(columns={
    'indice_calidad_global': 'calidad_global_lag1',
    'tasa_cortes_pct':       'tasa_cortes_lag1',
    'cobertura_5g_pct':      'cobertura_5g_lag1',
})
zona_cliente = clientes[['cliente_id', 'zona_id']]
panel = panel.merge(zona_cliente, on='cliente_id', how='left')
panel = panel.merge(calidad_lag, on=['zona_id', 'mes'], how='left')

# Perfil estatico
cols_perfil = [
    'cliente_id', 'edad', 'sexo', 'estado_civil', 'num_lineas',
    'tipo_plan', 'tipo_zona', 'region', 'ingreso_estimado',
    'antiguedad_meses', 'descuento_activo', 'tipo_dispositivo',
]
cols_perfil_disp = [c for c in cols_perfil if c in clientes.columns]
panel = panel.merge(clientes[cols_perfil_disp], on='cliente_id', how='left')
panel['tipo_plan_num'] = panel['tipo_plan'].map({'Basico': 1, 'Estandar': 2, 'Premium': 3})

# Merges V2
panel = panel.merge(sop_critico_lag, on=['cliente_id', 'mes'], how='left')
panel['critica_pendiente_lag1'] = panel['critica_pendiente_lag1'].fillna(0)

panel = panel.merge(
    clima_zonal[['zona_id', 'mes', 'sentimiento_zonal_lag1']],
    on=['zona_id', 'mes'], how='left'
)
panel = panel.merge(ratio_lag, on=['cliente_id', 'mes'], how='left')

print(f'Panel final: {panel.shape[0]:,} filas x {panel.shape[1]} columnas')


Panel final: 321,987 filas x 41 columnas


## 4. Seleccion de variables


In [ ]:
FEATURES_NUM = [
    # Perfil
    'edad', 'num_lineas', 'ingreso_estimado', 'antiguedad_meses', 'descuento_activo',
    # Facturacion lag 1
    'importe_mes_lag1', 'impago_mes_lag1', 'dias_retraso_mes_lag1',
    'stress_mes_lag1', 'consumo_extra_mes_lag1', 'variacion_consumo_lag1',
    # Rolling 3 meses
    'impago_mes_roll3_lag1', 'stress_mes_roll3_lag1',
    'variacion_consumo_roll3_lag1', 'dias_retraso_mes_roll3_lag1',
    # Features de tendencia (Opcion A)
    'tendencia_importe_lag1', 'tendencia_stress_lag1',
    'racha_impagos_lag1', 'sin_consumo_2m_lag1',
    'satisfaccion_hist6',
    # Soporte lag 1
    'n_interacciones_mes_lag1', 'n_baja_mes_lag1',
    # Calidad de red lag 1
    'calidad_global_lag1', 'tasa_cortes_lag1', 'cobertura_5g_lag1',
    # NUEVAS V2
    'critica_pendiente_lag1',
    'sentimiento_zonal_lag1',
    'ratio_estres_lag1',
]

FEATURES_CAT_NOMINAL = [
    'region', 'tipo_zona', 'sexo', 'estado_civil', 'tipo_dispositivo',
]

FEATURES_ORDINAL = ['tipo_plan_num']

TARGET       = 'churn'
all_features = FEATURES_NUM + FEATURES_CAT_NOMINAL + FEATURES_ORDINAL

missing = [f for f in all_features if f not in panel.columns]
if missing:
    print('Variables no encontradas:', missing)
else:
    print(f'OK - {len(all_features)} variables disponibles')
    print('Nuevas V2: critica_pendiente_lag1, sentimiento_zonal_lag1, ratio_estres_lag1')


OK - 32 variables disponibles
Nuevas V2: critica_pendiente_lag1, sentimiento_zonal_lag1, ratio_estres_lag1


## 5. Division train / test y preprocesado


In [ ]:
# Split por cliente (mismo criterio que modelos anteriores)
clientes_unicos = panel['cliente_id'].unique()
np.random.seed(RANDOM_STATE)
clientes_test  = np.random.choice(
    clientes_unicos, size=int(len(clientes_unicos) * 0.2), replace=False
)
clientes_train = np.setdiff1d(clientes_unicos, clientes_test)

train = panel[panel['cliente_id'].isin(clientes_train)].dropna(
    subset=['importe_mes_lag1', 'stress_mes_lag1']
).copy()
test = panel[panel['cliente_id'].isin(clientes_test)].dropna(
    subset=['importe_mes_lag1', 'stress_mes_lag1']
).copy()

X_train = train[all_features]
y_train = train[TARGET]
X_test  = test[all_features]
y_test  = test[TARGET]
RATIO   = round((y_train == 0).sum() / y_train.sum())

print(f'Train: {len(X_train):,} filas | {y_train.mean()*100:.2f}% churn')
print(f'Test:  {len(X_test):,} filas  | {y_test.mean()*100:.2f}% churn')
print(f'Ratio desbalance: {RATIO}:1')

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
nom_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='desconocido')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
])
ord_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, FEATURES_NUM),
    ('nom', nom_pipeline, FEATURES_CAT_NOMINAL),
    ('ord', ord_pipeline, FEATURES_ORDINAL),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print('Preprocesador definido')


Train: 257,596 filas | 0.63% churn
Test:  64,391 filas  | 0.63% churn
Ratio desbalance: 160:1
Preprocesador definido


## 6. Funcion de evaluacion


In [ ]:
def evaluar_modelo(nombre, y_true, y_pred_proba, umbral=0.05):
    y_pred = (y_pred_proba >= umbral).astype(int)
    return {
        'Modelo':    nombre,
        'AUC-ROC':   round(roc_auc_score(y_true, y_pred_proba), 3),
        'AUC-PR':    round(average_precision_score(y_true, y_pred_proba), 3),
        'Accuracy':  round(accuracy_score(y_true, y_pred), 3),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 3),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 3),
        'F1':        round(f1_score(y_true, y_pred, zero_division=0), 3),
    }

resultados = []
probas     = {}

# Referencia del mejor modelo anterior
resultados.append({
    'Modelo': 'LR GridSearch (anterior)',
    'AUC-ROC': 0.690, 'AUC-PR': 0.016,
    'Accuracy': None, 'Precision': None, 'Recall': None, 'F1': None,
})

print('Funcion de evaluacion definida')


Funcion de evaluacion definida


---
## 7. GridSearch - Los tres modelos con features V2


In [ ]:
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

model_configs = {
    'Regresion Logistica': {
        'estimator': Pipeline([
            ('preprocessor', preprocessor),
            ('model', LogisticRegression(
                class_weight='balanced', random_state=RANDOM_STATE,
                max_iter=1000, solver='saga'
            ))
        ]),
        'params':   {'model__C': [0.01, 0.1, 1.0], 'model__penalty': ['l1', 'l2']},
        'usa_prep': False,
    },
    'Random Forest': {
        'estimator': Pipeline([
            ('preprocessor', preprocessor),
            ('model', RandomForestClassifier(
                class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
            ))
        ]),
        'params': {
            'model__n_estimators':     [100, 200],
            'model__max_depth':        [5, 8],
            'model__min_samples_leaf': [20, 50],
        },
        'usa_prep': False,
    },
}

if XGBOOST_OK:
    model_configs['XGBoost'] = {
        'estimator': XGBClassifier(
            scale_pos_weight=RATIO, random_state=RANDOM_STATE,
            eval_metric='auc', verbosity=0
        ),
        'params': {
            'n_estimators':  [200, 400],
            'max_depth':     [4, 6],
            'learning_rate': [0.01, 0.05],
            'subsample':     [0.7, 0.9],
        },
        'usa_prep': True,
    }

best_estimators = {}

for nombre, config in model_configs.items():
    print(f'--- GridSearch: {nombre} ---')
    X_tr = X_train_prep if config['usa_prep'] else X_train
    X_te = X_test_prep  if config['usa_prep'] else X_test
    gs = GridSearchCV(
        config['estimator'], config['params'],
        cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1
    )
    gs.fit(X_tr, y_train)
    print(f'  Mejores parametros: {gs.best_params_}')
    print(f'  Mejor AUC en CV:    {gs.best_score_:.3f}')
    proba = gs.predict_proba(X_te)[:, 1]
    probas[nombre] = proba
    best_estimators[nombre] = gs.best_estimator_
    res = evaluar_modelo(nombre, y_test, proba)
    resultados.append(res)
    print(f'  AUC test: {res["AUC-ROC"]:.3f} | Recall: {res["Recall"]:.3f} | F1: {res["F1"]:.3f}')
    print()

print('GridSearch completado')


--- GridSearch: Regresion Logistica ---
Fitting 5 folds for each of 6 candidates, totalling 30 fits
  Mejores parametros: {'model__C': 0.01, 'model__penalty': 'l1'}
  Mejor AUC en CV:    0.716
  AUC test: 0.703 | Recall: 1.000 | F1: 0.013

--- GridSearch: Random Forest ---
Fitting 5 folds for each of 8 candidates, totalling 40 fits
  Mejores parametros: {'model__max_depth': 5, 'model__min_samples_leaf': 50, 'model__n_estimators': 200}
  Mejor AUC en CV:    0.702
  AUC test: 0.691 | Recall: 1.000 | F1: 0.013

--- GridSearch: XGBoost ---
Fitting 5 folds for each of 16 candidates, totalling 80 fits
  Mejores parametros: {'learning_rate': 0.01, 'max_depth': 4, 'n_estimators': 200, 'subsample': 0.7}
  Mejor AUC en CV:    0.704
  AUC test: 0.697 | Recall: 1.000 | F1: 0.013

GridSearch completado


---
## 8. Comparativa: mejoran las features V2?


In [ ]:
df_res = pd.DataFrame(resultados).set_index('Modelo')
print('=== COMPARATIVA ===')
display(
    df_res.style
    .highlight_max(subset=['AUC-ROC', 'AUC-PR', 'Recall', 'F1'], color='#c8f5c8')
    .format('{:.3f}', na_rep='-')
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colores = {
    'Regresion Logistica': '#4C9BE8',
    'Random Forest':       '#f39c12',
    'XGBoost':             '#E85C4C',
}
for nombre, proba in probas.items():
    RocCurveDisplay.from_predictions(
        y_test, proba, name=nombre,
        color=colores.get(nombre, 'gray'), ax=axes[0]
    )
axes[0].plot([0, 1], [0, 1], 'k--', label='Baseline (0.500)')
axes[0].set_title('Curvas ROC - Features V2', fontweight='bold')
axes[0].legend(fontsize=8)

for nombre, proba in probas.items():
    PrecisionRecallDisplay.from_predictions(
        y_test, proba, name=nombre,
        color=colores.get(nombre, 'gray'), ax=axes[1]
    )
axes[1].set_title('Curvas Precision-Recall', fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# Importancia de variables - LR (coeficientes)
if 'Regresion Logistica' in best_estimators:
    lr = best_estimators['Regresion Logistica']
    nom_names = (
        lr.named_steps['preprocessor']
        .named_transformers_['nom']
        .named_steps['encoder']
        .get_feature_names_out(FEATURES_CAT_NOMINAL)
    )
    feat_names = FEATURES_NUM + list(nom_names) + FEATURES_ORDINAL
    coefs = pd.Series(lr.named_steps['model'].coef_[0], index=feat_names)
    top20 = coefs.sort_values(key=abs, ascending=False).head(20).sort_values()

    fig, ax = plt.subplots(figsize=(9, 7))
    colors = ['#E85C4C' if v > 0 else '#4C9BE8' for v in top20.values]
    ax.barh(top20.index, top20.values, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(
        'LR Features V2 - Top 20 variables\n'
        '(rojo = aumenta churn | azul = lo reduce)',
        fontweight='bold'
    )
    ax.set_xlabel('Coeficiente (escala estandarizada)')

    nuevas_v2   = ['critica_pendiente_lag1', 'sentimiento_zonal_lag1', 'ratio_estres_lag1']
    nuevas_prev = [
        'tendencia_importe_lag1', 'tendencia_stress_lag1',
        'racha_impagos_lag1', 'sin_consumo_2m_lag1', 'satisfaccion_hist6',
    ]
    for label in ax.get_yticklabels():
        if label.get_text() in nuevas_v2:
            label.set_color('#9b59b6')
            label.set_fontweight('bold')
        elif label.get_text() in nuevas_prev:
            label.set_color('#27ae60')
            label.set_fontweight('bold')

    ax.text(0.98, 0.06, '* Variables nuevas V2 en morado',
            transform=ax.transAxes, ha='right', fontsize=8, color='#9b59b6')
    ax.text(0.98, 0.02, '* Variables Opcion A en verde',
            transform=ax.transAxes, ha='right', fontsize=8, color='#27ae60')
    plt.tight_layout()
    plt.show()


---
## 9. Conclusiones

### Mejoran las features V2 el AUC respecto a Opcion A?

**Si, aunque de forma marginal.** El mejor modelo (LR GridSearch) pasa de **AUC 0.701 a 0.703**, con mejora tambien en CV (0.714 a 0.716).

Las tres variables nuevas aportan de forma diferente:

- **`ratio_estres_lag1`** entra en el **top 3** de coeficientes, confirmando la hipotesis de multicausalidad: la combinacion de mala red + retraso en el pago es mas predictiva que cada variable por separado.
- **`critica_pendiente_lag1`** aparece en el top 10, validando que una incidencia critica sin resolver el mes anterior es una senal adelantada de churn real.
- **`sentimiento_zonal_lag1`** no entra en el top 20. Consistente con el EDA de encuestas: el NPS zonal correlaciona con churn agregado por zona, pero no aporta senal incremental cuando ya tenemos `stress_mes` y `calidad_global`.

### Comparativa global del proyecto

| Modelo | AUC | Features | Leakage | Produccion |
|--------|-----|----------|---------|------------|
| Binario LR | 0.991 | Basicas | Si | No |
| Binario RF | 0.994 | Basicas | Si | No |
| Temporal LR | 0.685 | Basicas | No | Si |
| LR GridSearch | 0.690 | Basicas | No | Si |
| LR Opcion A | 0.701 | + Tendencia | No | Si |
| **LR Features V2** | **0.703** | **+ V2** | **No** | **Si - MEJOR** |

### Reflexion sobre el techo del modelo

Que tres modelos muy distintos (LR, RF, XGBoost) converjan en el rango 0.691-0.703 sugiere que estamos cerca del **techo informativo** de los datos disponibles. El churn tiene una componente aleatoria o depende de factores no observados (oferta de la competencia, cambio de situacion personal) que ningun modelo puede capturar. Un AUC de 0.703 en produccion, con desbalance 160:1 y sin leakage, es un resultado solido y realista.

---
*Prediccion de Churn - Empresa de Telecomunicaciones | Practicas Aplicadas 2026*
